# HelioAI — Jupyter tour

A guided tour of HelioAI inside a notebook: ask questions in natural language, get
figures inline, and keep working on the generated code in the same session.

**Before running this**

1. `pip install helioai` (or `uv sync` from a clone)
2. Set one LLM provider key in `.env` — see the
   [installation guide](https://erdoganfurkan.github.io/HelioAI/installation/)
3. Build the parameter index once: `helioai index` (~10 min, 83k products)

Data access itself needs no credentials.

> Outputs are stripped from this file on purpose — run the cells to produce your own.


In [ ]:
%load_ext helioai.interfaces.jupyter_magic

import os

provider = os.environ.get("HELIOAI_LLM_PROVIDER", "azure")
print(f"Provider : {provider}")
print("Ready — use %%helioai in any cell below.")

---
## 1 — Parameter Discovery: Cluster C3 Magnetopause Crossing

> **Event:** Retinò et al. 2007, *Nature Physics* 3, 235–238 — magnetic reconnection at the magnetopause  
> **Window:** `2007-03-05T18:00:00Z` → `2007-03-05T20:00:00Z`

The `parameter_hunter` sub-agent resolves ambiguous parameter names against the 65k-entry catalog.
Classic pitfalls: `C1` vs `C3`, `onboard` vs `prime` moments, `CIS-HIA` vs `CIS-CODIF`.

**Expected:** ion density ~20–50 cm⁻³ in magnetosheath, ~0.1–1 cm⁻³ inside the magnetosphere.

In [ ]:
%%helioai
I'm looking at a magnetopause crossing by Cluster C3 on March 5, 2007
around 18:00–20:00 UT — this is the Retinò et al. 2007 Nature Physics
reconnection event. I need the ion number density and bulk velocity
from the CIS-HIA instrument. Can you find me the right speasy parameter
ids? I'm not sure whether to use onboard moments or prime parameters,
and whether the spacecraft is labelled C3 or Cluster-3 in the catalog.

---
## 2 — Solar Wind Analysis: Halloween Storm 2003

> **Event:** Skoug et al. 2004, *JGR* 109, A09102 — Dst_min = −363 nT  
> **Window:** `2003-10-29T05:00:00Z` → `2003-10-30T12:00:00Z`

The `data_analyst` sub-agent downloads ACE data, marks the shock at 06:11 UT, and computes sheath statistics.

**Expected output:**
- Three-panel plot: IMF Bz, proton density, bulk speed
- Shock annotation at ~06:11 UT
- Bz_min ~ −35 nT, density peak ~50–90 cm⁻³ in the sheath

In [ ]:
%%helioai
Plot the solar wind IMF Bz, proton density and bulk speed from ACE
during the Halloween storm: 2003-10-29T05:00:00 to 2003-10-30T12:00:00.
I want to see the shock arrival around 06:11 UT on the 29th, the sheath
with variable Bz, and the main southward turning that drove the Dst to
−363 nT. Mark the shock time on the plot and compute the minimum Bz
and peak density in the sheath.

---
## 3 — Plasma Physics: CIR/HSS Interface September 2017

> **Event:** Co-rotating Interaction Region — stream interface  
> **Window:** `2017-09-27T00:00:00Z` → `2017-09-28T12:00:00Z`

The `plasma_physicist` sub-agent computes time-varying plasma parameters with PlasmaPy across the CIR compression ridge.

**Reference values at the interface:**

| Quantity | Value |
|---|---|
| Plasma β | 1–3 (compressed region) |
| Alfvén speed | 60–80 km/s |
| f_ci at 20 nT | ~0.30 Hz |
| Solar wind speed | 350 → 720 km/s ramp |

In [ ]:
%%helioai
Compute the proton plasma beta, Alfvén speed, and proton gyrofrequency
across the CIR pressure ridge on 2017-09-27T00:00:00 to
2017-09-28T12:00:00 using ACE or Wind data. I expect the solar wind
speed to ramp from ~350 to ~720 km/s at the stream interface, with
compressed density ~15–20 cm⁻³ and |B| up to 20 nT. Show me how
these quantities evolve across the interface and flag where β > 1.

---
## 4 — Shock Detection: IP Shock 2005-01-18 ✅ Validated

> **Event:** CME from X3.8 flare, AR 10720, January 17 2005  
> **Reference:** Wind MFI IP Shock Catalog (wind.nasa.gov/mfi/ip_shock.html)  
> **Window:** `2005-01-17T12:00:00Z` → `2005-01-18T10:00:00Z`

This is our scored benchmark. Previous run result:

| Metric | Agent | Reference | Status |
|---|---|---|---|
| Shock time | 06:53 UT | ~06:50 UT | ✅ +3 min |
| Compression ratio | 2.97× | 3–5× | ✅ |
| Score | **8/10** | — | |

**Expected plot:** density, speed, and |B| with a sharp jump marking the shock arrival.

In [ ]:
%%helioai
Detect the interplanetary shock arrival at ACE on 2005-01-18 — a fast
CME launched by the X3.8 flare from AR 10720 on January 17. The shock
is catalogued in the Wind MFI IP shock list, expected around 06:00–07:00
UT on the 18th. Use ACE solar wind data from 2005-01-17T12:00:00 to
2005-01-18T10:00:00, detect the pressure jump, give me the exact arrival
timestamp and the density compression ratio.

---
## 5 — Multi-spacecraft: L1 Constellation November 2017

> **Event:** Burkholder et al. 2020, *JGR Space Physics* 125, e2020JA027978  
> **Window:** `2017-11-17T00:00:00Z` → `2017-11-17T12:00:00Z`

The `cross_mission` sub-agent fetches IMF Bz simultaneously from ACE, Wind, and DSCOVR, then computes cross-correlation lags to infer the solar wind front propagation direction.

**Expected:**
- Overlaid three-spacecraft Bz time series
- Propagation lag ~1–5 min between pairs
- Solar wind speed ~400–500 km/s on this day

In [ ]:
%%helioai
Compare the IMF Bz from ACE, Wind and DSCOVR on 2017-11-17T00:00:00
to 2017-11-17T12:00:00. This is a Burkholder et al. 2020 case study
for L1 multi-spacecraft monitoring — the three spacecraft have a
measurable timing offset reflecting their ~2–3° angular separation.
Compute the cross-correlation lag between each pair and tell me the
inferred propagation delay in minutes. Plot the three time series
overlaid.

---
## 6 — Direct PlasmaPy Calculations (instant, no LLM)

For quick sanity checks, call the plasma physics tools directly — zero latency, no API call.

Useful pattern: compute reference values *before* asking the agent, then compare.

In [ ]:
from helioai.tools.plasmapy_tools import alfven_speed, debye_length, gyrofrequency, plasma_beta

# Typical values for three plasma regimes
regimes = [
    ("Quiet solar wind", {"B_nT": 5.0, "n_cm3": 5.0, "T_eV": 10.0}),
    ("CIR compression", {"B_nT": 20.0, "n_cm3": 15.0, "T_eV": 20.0}),
    ("Magnetosheath", {"B_nT": 20.0, "n_cm3": 25.0, "T_eV": 100.0}),
]

print(f"{'Region':<22} {'β':>6}  {'V_A (km/s)':>10}  {'f_ci (Hz)':>9}  {'λ_D (m)':>8}")
print("-" * 65)
for label, p in regimes:
    b = await plasma_beta(p["B_nT"], p["n_cm3"], p["T_eV"])
    va = await alfven_speed(p["B_nT"], p["n_cm3"])
    fci = await gyrofrequency(p["B_nT"], "proton")
    ld = await debye_length(p["n_cm3"], p["T_eV"])
    print(
        f"{label:<22} {b['beta']:>6.2f}  "
        f"{va['alfven_speed_km_s']:>10.1f}  "
        f"{fci['frequency_Hz']:>9.4f}  "
        f"{ld['debye_length_m']:>8.2f}"
    )

In [ ]:
# Power spectrum helper — demo on synthetic B data
import numpy as np

from helioai.tools.plasmapy_tools import power_spectrum

# Synthetic IMF with two injected frequencies: 0.1 Hz and 0.3 Hz
dt = 0.5  # 2 Hz cadence
t = np.arange(0, 300, dt)
B = (
    np.sin(2 * np.pi * 0.10 * t)
    + 0.5 * np.sin(2 * np.pi * 0.30 * t)
    + 0.1 * np.random.randn(len(t))
)

result = await power_spectrum(B.tolist(), dt_s=dt)
print(f"Peak frequency : {result['peak_frequency_Hz']:.3f} Hz")
print(f"Peak power     : {result['peak_power']:.3f} (arb. units)")

---
## 7 — Event Catalogs

The strongest reason to use HelioAI rather than downloading by hand: run the *same*
analysis across every event in a curated catalog. 217 AMDA catalogs and timetables are
available — ICMEs, bow-shock crossings, reconnection events, substorm onsets.


In [ ]:
%%helioai
What event catalogs are available for interplanetary shocks?


In [ ]:
%%helioai
Using the Richardson & Cane ICME catalog, take the events between 2003 and 2005
and run a superposed epoch analysis of the IMF Bz around each arrival.


---
## 8 — Session History & Resume

All conversations are persisted to SQLite (`data/sessions.db`).  
Use `%helioai_history` to browse past sessions and `%helioai_resume` to continue one.

In [ ]:
# Browse sessions for this Jupyter user
%helioai_history

In [ ]:
# Resume a previous session using the first 8 chars of its id:
# %helioai_resume abc12345

# Start fresh:
%helioai_session reset

---
## 9 — Where to go next

| Interface | How |
|---|---|
| Interactive CLI | `helioai` |
| One-shot | `helioai "your query"` |
| Jupyter | this notebook |
| Web UI | `helioai serve --web` → http://localhost:7890 |
| MCP server | `helioai-mcp`, or `--http` |

Export this session as a standalone notebook — `load_data()` becomes direct
`spz.get_data(...)`, sandbox-only helpers are stripped, and a *Methods & data
acknowledgements* cell lists every recipe and reference used:

```python
%helioai_export
```

**Links**

- Documentation: <https://erdoganfurkan.github.io/HelioAI>
- speasy: <https://github.com/SciQLop/speasy>
- PlasmaPy: <https://docs.plasmapy.org>
- AMDA: <https://amda.irap.omp.eu>
- CDAWeb: <https://cdaweb.gsfc.nasa.gov>
